In [2]:
import os
import json
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Path setup
CSV_PATH = "data/processed/cases.csv"
EVAL_DIR = "data/eval"
os.makedirs(EVAL_DIR, exist_ok=True)

# BACA DATASET (Sudah diperbaiki menggunakan read_csv)
df = pd.read_csv(CSV_PATH)
print(f"✅ Berhasil memuat {len(df)} data kasus hak cipta.")

✅ Berhasil memuat 37 data kasus hak cipta.


In [3]:
# Melakukan splitting data menjadi data train dan data test dengan rasio 80:20
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

print(f"📊 Hasil Splitting Data:")
print(f"🔹 Jumlah Data Train (Basis Kasus): {len(df_train)} kasus")
print(f"🔹 Jumlah Data Test (Uji Model): {len(df_test)} kasus")

📊 Hasil Splitting Data:
🔹 Jumlah Data Train (Basis Kasus): 29 kasus
🔹 Jumlah Data Test (Uji Model): 8 kasus


In [4]:
# Inisialisasi TfidfVectorizer untuk pembobotan kata kunci fakta hukum
tfidf = TfidfVectorizer(stop_words='english')

# Fitting dan transformasi pada teks kronologi/fakta kasus train
tfidf_matrix_train = tfidf.fit_transform(df_train['ringkasan_fakta'].fillna(''))

print("📐 Matriks Vektor TF-IDF untuk basis kasus berhasil dibangun!")

📐 Matriks Vektor TF-IDF untuk basis kasus berhasil dibangun!


In [8]:
def retrieve(query: str, k: int = 5):
    # 1) Pre-process query (diubah ke lowercase)
    query_clean = query.lower()
    
    # 2) Hitung vektor TF-IDF untuk query kasus baru
    query_vector = tfidf.transform([query_clean])
    
    # 3) Hitung Cosine Similarity antara query dengan seluruh basis kasus train
    similarity_scores = cosine_similarity(query_vector, tfidf_matrix_train).flatten()
    
    # Ambil index top-k similarity tertinggi
    # BENAR
    top_k_indices = similarity_scores.argsort()[-k:][::-1]
    
    # Bangun list hasil retrieval
    results = []
    for rank, idx in enumerate(top_k_indices, start=1):
        matched_case = df_train.iloc[idx]
        results.append({
            "rank": rank,
            "case_id": int(matched_case['case_id']),
            "no_perkara": matched_case['no_perkara'],
            "pihak": matched_case['pihak'],
            "similarity": round(float(similarity_scores[idx]), 4)
        })
        
    return results

print("✅ Fungsi retrieve() sesuai standar siklus CBR berhasil diimplementasikan.")

✅ Fungsi retrieve() sesuai standar siklus CBR berhasil diimplementasikan.


In [9]:
# Ambil beberapa contoh fakta hukum dari data test untuk dijadikan bahan query uji coba
sample_queries = []
for idx, row in df_test.head(5).iterrows():
    sample_queries.append({
        "query_id": int(row['case_id']),
        "query_text": row['ringkasan_fakta'][:300], # Potongan fakta sebagai input query
        "ground_truth_case_id": int(row['case_id']) # Label asli untuk evaluasi akurasi nanti
    })

# Simpan ke /data/eval/queries.json sesuai instruksi tugas
queries_json_path = os.path.join(EVAL_DIR, "queries.json")
with open(queries_json_path, 'w', encoding='utf-8') as f:
    json.dump(sample_queries, f, indent=4, ensure_ascii=False)

print(f"🎉 File pengujian sukses dibuat di: {queries_json_path}")

🎉 File pengujian sukses dibuat di: data/eval\queries.json


In [10]:
# Uji coba memasukkan contoh query kasus pembajakan acak
contoh_query = "terdakwa terbukti dengan sengaja mengedarkan, memajang, dan menjual kaset vcd bajakan mp3 lagu tanpa izin"
hasil_pencarian = retrieve(contoh_query, k=3)

print(f"🔎 Hasil Pencarian Kasus Terkait untuk Query: '{contoh_query}'\n")
for res in hasil_pencarian:
    print(f"Rank {res['rank']} -> [Case ID: {res['case_id']}] {res['no_perkara']} | Pihak: {res['pihak']} | Skor Kemiripan: {res['similarity']}")

🔎 Hasil Pencarian Kasus Terkait untuk Query: 'terdakwa terbukti dengan sengaja mengedarkan, memajang, dan menjual kaset vcd bajakan mp3 lagu tanpa izin'

Rank 1 -> [Case ID: 37] 1813 k/pid.sus/2009 | Pihak: a d i y a n t o. | Skor Kemiripan: 0.2504
Rank 2 -> [Case ID: 1] 1813 k/pid.sus/2009 | Pihak: a d i y a n t o. | Skor Kemiripan: 0.2504
Rank 3 -> [Case ID: 31] 1101 k/pid.sus/2016 | Pihak: hi. achmad budi siswanto s.h. bin muchsin jatno | Skor Kemiripan: 0.1451
